In [1]:
from core.libs import pd, np
from core.db import get_engine

engine = get_engine()

df_events = pd.read_excel(
    r"C:\Users\HeyCe\World Tree Technologies Inc\Operations - Documentos\WorldTreeSystem\DatabaseExports\inventories_tree_age.xlsx")


💻 Conectado a la base de datos helloworldtree


In [2]:
# ARREGLA ESTA CELDA:
query = """
SELECT
    cti.contract_code,
    cti.farmer_number,
    fpi.contract_name,
    cti.planting_year,     -- ✅ YA ESTÁ AQUÍ
    cti.status              -- ✅ YA ESTÁ AQUÍ
FROM masterdatabase.contract_tree_information cti
LEFT JOIN masterdatabase.farmer_personal_information fpi
    ON cti.farmer_number = fpi.farmer_number
"""
df_cti = pd.read_sql(query, engine)

# NO ELIMINES ESTAS COLUMNAS:
df_cti = df_cti[[
    "contract_code",
    "farmer_number",
    "contract_name",
    "planting_year",   # ⚠️ MANTÉN ESTA
    "status"           # ⚠️ MANTÉN ESTA
]]



In [3]:
df_events = df_events.merge(
    df_cti[["contract_code"]],
    on="contract_code",
    how="left",
    validate="many_to_one"
)


In [4]:
# 1) Eventos de Inventory con edad numérica confiable
inv_events = df_events.copy()
inv_events["tree_age"] = pd.to_numeric(inv_events["tree_age"], errors="coerce")

inv_events = inv_events[
    (inv_events["inventory_type"] == "Inventory") &
    (inv_events["tree_age"].notna())
].copy()

# 2) Asignar milestone bucket según edad
inv_events["inventory_milestone"] = pd.cut(
    inv_events["tree_age"],
    bins=[3, 6, 9, np.inf],
    labels=["Inventory1", "Inventory2", "Inventory3"],
    right=False
)

# 3) Flags por contrato
milestone_flags = (
    inv_events
    .dropna(subset=["inventory_milestone"])
    .pivot_table(
        index="contract_code",
        columns="inventory_milestone",
        values="tree_age",
        aggfunc="size",
        fill_value=0
    )
    .gt(0)
    .reset_index()
)

for col in ["Inventory1", "Inventory2", "Inventory3"]:
    if col not in milestone_flags.columns:
        milestone_flags[col] = False

milestone_flags = milestone_flags.rename(columns={
    "Inventory1": "has_inventory1",
    "Inventory2": "has_inventory2",
    "Inventory3": "has_inventory3",
})


C:\Users\HeyCe\AppData\Local\Temp\ipykernel_9184\605898806.py:22: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


In [5]:
df_base = df_cti.merge(
    milestone_flags,
    on="contract_code",
    how="left"
)

for c in ["has_inventory1", "has_inventory2", "has_inventory3"]:
    df_base[c] = df_base[c].fillna(False)


C:\Users\HeyCe\AppData\Local\Temp\ipykernel_9184\1053010448.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_base[c] = df_base[c].fillna(False)


In [6]:
df_base_active = df_base[df_base["status"] == "Active"].copy()


In [7]:
# 0) FILTRO BASE (solo Active)
df_inventory_plan_2026 = df_base_active.copy()

df_inventory_plan_2026["tree_age_2026"] = 2026 - df_inventory_plan_2026["planting_year"]

df_inventory_plan_2026["inventory_2026"] = False
df_inventory_plan_2026["expected_inventory_type_2026"] = None
df_inventory_plan_2026["rule_applied"] = None



In [8]:
mask = df_inventory_plan_2026["planting_year"] == 2025
df_inventory_plan_2026.loc[mask, [
    "inventory_2026",
    "expected_inventory_type_2026",
    "rule_applied"
]] = [True, "Survival0", "PY_2025_SURVIVAL0"]


In [9]:
mask = df_inventory_plan_2026["planting_year"] == 2024
df_inventory_plan_2026.loc[mask, [
    "inventory_2026",
    "expected_inventory_type_2026",
    "rule_applied"
]] = [True, "Survival1", "PY_2024_SURVIVAL1"]


In [10]:
mask = df_inventory_plan_2026["planting_year"] == 2023
df_inventory_plan_2026.loc[mask, [
    "inventory_2026",
    "expected_inventory_type_2026",
    "rule_applied"
]] = [True, "Inventory1", "PY_2023_INVENTORY1"]


In [11]:
df_base = df_base.merge(milestone_flags, on="contract_code", how="left")

for c in ["has_inventory1", "has_inventory2", "has_inventory3"]:
    df_base[c] = df_base[c].fillna(False)


KeyError: 'has_inventory1'

In [1]:
from core.libs import pd, np
from core.db import get_engine

# ============================================================================
# 1. CARGAR DATOS BASE
# ============================================================================
engine = get_engine()

# Eventos de inventarios históricos
df_events = pd.read_excel(
    r"C:\Users\HeyCe\World Tree Technologies Inc\Operations - Documentos\WorldTreeSystem\DatabaseExports\inventories_tree_age.xlsx"
)

# Información de contratos
query = """
SELECT
    cti.contract_code,
    cti.farmer_number,
    fpi.contract_name,
    cti.planting_year,
    cti.status
FROM masterdatabase.contract_tree_information cti
LEFT JOIN masterdatabase.farmer_personal_information fpi
    ON cti.farmer_number = fpi.farmer_number
"""
df_cti = pd.read_sql(query, engine)

# ============================================================================
# 2. PROCESAR INVENTARIOS HISTÓRICOS Y EXTRAER ÚLTIMOS AÑOS
# ============================================================================
inv_events = df_events.copy()
inv_events["tree_age"] = pd.to_numeric(inv_events["tree_age"], errors="coerce")

# Filtrar solo Inventories con edad válida
inv_events = inv_events[
    (inv_events["inventory_type"] == "Inventory") &
    (inv_events["tree_age"].notna())
].copy()

# Clasificar por milestone según edad
inv_events["inventory_milestone"] = pd.cut(
    inv_events["tree_age"],
    bins=[3, 6, 9, np.inf],
    labels=["Inventory1", "Inventory2", "Inventory3"],
    right=False
)

# Obtener el ÚLTIMO AÑO de cada tipo de inventario por contrato
last_inventory_years = (
    inv_events
    .dropna(subset=["inventory_milestone"])
    .groupby(["contract_code", "inventory_milestone"])["inventory_year"]
    .max()
    .reset_index()
    .pivot(index="contract_code", columns="inventory_milestone", values="inventory_year")
    .reset_index()
)

# Renombrar columnas
last_inventory_years.columns.name = None
for col in ["Inventory1", "Inventory2", "Inventory3"]:
    if col in last_inventory_years.columns:
        last_inventory_years = last_inventory_years.rename(columns={
            col: f"last_{col.lower()}_year"
        })
    else:
        last_inventory_years[f"last_{col.lower()}_year"] = pd.NA

# ============================================================================
# 3. CONSTRUIR TABLA BASE
# ============================================================================
df_base = df_cti.merge(
    last_inventory_years,
    on="contract_code",
    how="left"
)

# Filtrar solo contratos activos
df_base_active = df_base[df_base["status"] == "Active"].copy()

# ============================================================================
# 4. CALCULAR EDAD EN 2026 Y CLASIFICACIÓN
# ============================================================================
df_output = df_base_active.copy()
df_output["tree_age_2026"] = 2026 - df_output["planting_year"]

# Clasificar según edad en 2026
df_output["age_classification_2026"] = pd.cut(
    df_output["tree_age_2026"],
    bins=[-np.inf, 1, 2, 3, 6, 9, np.inf],
    labels=["Survival0", "Survival1", "Survival2", "Inventory1", "Inventory2", "Inventory3"],
    right=False
)

# ============================================================================
# 5. DETERMINAR PLAN DE INVENTARIO 2026
# ============================================================================
df_output["inventory_2026"] = False
df_output["expected_inventory_type_2026"] = None
df_output["rule_applied"] = None

# Regla 1: Planting Year 2025 → Survival0
mask = df_output["planting_year"] == 2025
df_output.loc[mask, ["inventory_2026", "expected_inventory_type_2026", "rule_applied"]] = [
    True, "Survival0", "PY_2025_SURVIVAL0"
]

# Regla 2: Planting Year 2024 → Survival1
mask = df_output["planting_year"] == 2024
df_output.loc[mask, ["inventory_2026", "expected_inventory_type_2026", "rule_applied"]] = [
    True, "Survival1", "PY_2024_SURVIVAL1"
]

# Regla 3: Planting Year 2023 → Inventory1
mask = df_output["planting_year"] == 2023
df_output.loc[mask, ["inventory_2026", "expected_inventory_type_2026", "rule_applied"]] = [
    True, "Inventory1", "PY_2023_INVENTORY1"
]

# Regla 4: Árboles con 6-9 años en 2026 que NO tienen Inventory2
mask = (
    (df_output["tree_age_2026"] >= 6) &
    (df_output["tree_age_2026"] < 9) &
    (df_output["last_inventory2_year"].isna())
)
df_output.loc[mask, ["inventory_2026", "expected_inventory_type_2026", "rule_applied"]] = [
    True, "Inventory2", "MISSING_INVENTORY2"
]

# Regla 5: Árboles con 9+ años en 2026 que NO tienen Inventory3
mask = (
    (df_output["tree_age_2026"] >= 9) &
    (df_output["last_inventory3_year"].isna())
)
df_output.loc[mask, ["inventory_2026", "expected_inventory_type_2026", "rule_applied"]] = [
    True, "Inventory3", "MISSING_INVENTORY3"
]

# ============================================================================
# 6. ORDENAR COLUMNAS FINALES
# ============================================================================
df_output = df_output[[
    "contract_code",
    "farmer_number",
    "contract_name",
    "planting_year",
    "status",
    "tree_age_2026",
    "age_classification_2026",
    "last_inventory1_year",
    "last_inventory2_year",
    "last_inventory3_year",
    "inventory_2026",
    "expected_inventory_type_2026",
    "rule_applied"
]]

# ============================================================================
# 7. MOSTRAR RESULTADOS
# ============================================================================
print("=" * 80)
print("RESUMEN DEL PLAN DE INVENTARIOS 2026")
print("=" * 80)
print(f"Total contratos activos: {len(df_output)}")
print(f"Contratos que necesitan inventario en 2026: {df_output['inventory_2026'].sum()}")
print("\nDistribución por tipo de inventario esperado:")
print(df_output[df_output["inventory_2026"]]["expected_inventory_type_2026"].value_counts())
print("\n" + "=" * 80)

# Ver algunos ejemplos
print("\nEjemplos de contratos que necesitan inventario en 2026:")
print(df_output[df_output["inventory_2026"]].head(10))

💻 Conectado a la base de datos helloworldtree
RESUMEN DEL PLAN DE INVENTARIOS 2026
Total contratos activos: 265
Contratos que necesitan inventario en 2026: 179

Distribución por tipo de inventario esperado:
expected_inventory_type_2026
Inventory2    74
Survival1     63
Inventory1    31
Survival0      6
Inventory3     5
Name: count, dtype: int64


Ejemplos de contratos que necesitan inventario en 2026:
   contract_code farmer_number                        contract_name  \
0         US0170         10132               Shady Branch Farms LLC   
3         US0166         10130                          Ken English   
4         US0167         10067         Mike Porter and Teri Galardi   
5         US0168         10133  Justin Montgomery and Rusty Elliott   
6         CR0112         20103           Manfred Goyenaga Rodriguez   
8         US0169         10134        Corley Walton Family Farm LLC   
9         MX0071         40066                   Emma Barrios Goméz   
10        US0164         10

C:\Users\HeyCe\AppData\Local\Temp\ipykernel_21868\3175395592.py:52: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["contract_code", "inventory_milestone"])["inventory_year"]


In [1]:
from core.libs import pd, np
from core.db import get_engine

# ============================================================================
# 1. CARGAR DATOS BASE
# ============================================================================
engine = get_engine()

# Eventos de inventarios históricos
df_events = pd.read_excel(
    r"C:\Users\HeyCe\World Tree Technologies Inc\Operations - Documentos\WorldTreeSystem\DatabaseExports\inventories_tree_age.xlsx"
)

# Información de contratos
query = """
SELECT
    cti.contract_code,
    cti.farmer_number,
    fpi.contract_name,
    cti.planting_year,
    cti.status
FROM masterdatabase.contract_tree_information cti
LEFT JOIN masterdatabase.farmer_personal_information fpi
    ON cti.farmer_number = fpi.farmer_number
"""
df_cti = pd.read_sql(query, engine)

# ============================================================================
# 2. PROCESAR INVENTARIOS HISTÓRICOS Y EXTRAER ÚLTIMOS AÑOS
# ============================================================================
inv_events = df_events.copy()
inv_events["tree_age"] = pd.to_numeric(inv_events["tree_age"], errors="coerce")

# Filtrar solo Inventories con edad válida
inv_events = inv_events[
    (inv_events["inventory_type"] == "Inventory") &
    (inv_events["tree_age"].notna())
].copy()

# Clasificar por milestone según edad
inv_events["inventory_milestone"] = pd.cut(
    inv_events["tree_age"],
    bins=[3, 6, 9, np.inf],
    labels=["Inventory1", "Inventory2", "Inventory3"],
    right=False
)

# Obtener el ÚLTIMO AÑO de cada tipo de inventario por contrato
last_inventory_years = (
    inv_events
    .dropna(subset=["inventory_milestone"])
    .groupby(["contract_code", "inventory_milestone"])["inventory_year"]
    .max()
    .reset_index()
    .pivot(index="contract_code", columns="inventory_milestone", values="inventory_year")
    .reset_index()
)

# Renombrar columnas
last_inventory_years.columns.name = None
for col in ["Inventory1", "Inventory2", "Inventory3"]:
    if col in last_inventory_years.columns:
        last_inventory_years = last_inventory_years.rename(columns={
            col: f"last_{col.lower()}_year"
        })
    else:
        last_inventory_years[f"last_{col.lower()}_year"] = pd.NA

# ============================================================================
# 3. CONSTRUIR TABLA BASE
# ============================================================================
df_base = df_cti.merge(
    last_inventory_years,
    on="contract_code",
    how="left"
)

# Filtrar solo contratos activos
df_base_active = df_base[df_base["status"] == "Active"].copy()

# ============================================================================
# 4. CALCULAR EDAD EN 2026 Y CLASIFICACIÓN
# ============================================================================
df_output = df_base_active.copy()
df_output["tree_age_2026"] = 2026 - df_output["planting_year"]

# Clasificar según edad en 2026
df_output["age_classification_2026"] = pd.cut(
    df_output["tree_age_2026"],
    bins=[-np.inf, 1, 2, 3, 6, 9, np.inf],
    labels=["Survival0", "Survival1", "Survival2", "Inventory1", "Inventory2", "Inventory3"],
    right=False
)

# ============================================================================
# 5. DETERMINAR PLAN DE INVENTARIO 2026
# ============================================================================
df_output["inventory_2026"] = False
df_output["expected_inventory_type_2026"] = None
df_output["rule_applied"] = None

# Regla 1: Planting Year 2025 → Survival0
mask = df_output["planting_year"] == 2025
df_output.loc[mask, ["inventory_2026", "expected_inventory_type_2026", "rule_applied"]] = [
    True, "Survival0", "PY_2025_SURVIVAL0"
]

# Regla 2: Planting Year 2024 → Survival1
mask = df_output["planting_year"] == 2024
df_output.loc[mask, ["inventory_2026", "expected_inventory_type_2026", "rule_applied"]] = [
    True, "Survival1", "PY_2024_SURVIVAL1"
]

# Regla 3: Planting Year 2023 → Inventory1 (SOLO si NO existe Inventory1)
mask = (
    (df_output["planting_year"] == 2023) &
    (df_output["last_inventory1_year"].isna())
)
df_output.loc[mask, ["inventory_2026", "expected_inventory_type_2026", "rule_applied"]] = [
    True, "Inventory1", "PY_2023_INVENTORY1"
]

# Regla 3b: Planting Year 2023 pero YA tiene Inventory1 (drift detectado)
mask = (
    (df_output["planting_year"] == 2023) &
    (df_output["last_inventory1_year"].notna())
)
df_output.loc[mask, "rule_applied"] = "DRIFT_INVENTORY1_ALREADY_DONE"

# Regla 4: Planting Year 2020 → Inventory2 (SOLO si NO existe Inventory2)
mask = (
    (df_output["planting_year"] == 2020) &
    (df_output["last_inventory2_year"].isna())
)
df_output.loc[mask, ["inventory_2026", "expected_inventory_type_2026", "rule_applied"]] = [
    True, "Inventory2", "PY_2020_INVENTORY2"
]

# Regla 4b: Planting Year 2020 pero YA tiene Inventory2 (drift detectado)
mask = (
    (df_output["planting_year"] == 2020) &
    (df_output["last_inventory2_year"].notna())
)
df_output.loc[mask, "rule_applied"] = "DRIFT_INVENTORY2_ALREADY_DONE"

# Regla 5: Planting Year 2017 → Inventory3 (SOLO si NO existe Inventory3)
mask = (
    (df_output["planting_year"] == 2017) &
    (df_output["last_inventory3_year"].isna())
)
df_output.loc[mask, ["inventory_2026", "expected_inventory_type_2026", "rule_applied"]] = [
    True, "Inventory3", "PY_2017_INVENTORY3"
]

# Regla 5b: Planting Year 2017 pero YA tiene Inventory3 (drift detectado)
mask = (
    (df_output["planting_year"] == 2017) &
    (df_output["last_inventory3_year"].notna())
)
df_output.loc[mask, "rule_applied"] = "DRIFT_INVENTORY3_ALREADY_DONE"

# ============================================================================
# 7. AGREGAR COLUMNAS DE ANÁLISIS DE DRIFT
# ============================================================================
# Calcular si hubo drift (diferencia entre año esperado y año real)
df_output["inventory1_drift"] = None
df_output["inventory2_drift"] = None
df_output["inventory3_drift"] = None

# Drift Inventory1: esperado en tree_age=3 (2026 - PY + 3)
mask = df_output["last_inventory1_year"].notna()
expected_year_inv1 = df_output["planting_year"] + 3
df_output.loc[mask, "inventory1_drift"] = (
    df_output.loc[mask, "last_inventory1_year"] - expected_year_inv1[mask]
)

# Drift Inventory2: esperado en tree_age=6 (2026 - PY + 6)
mask = df_output["last_inventory2_year"].notna()
expected_year_inv2 = df_output["planting_year"] + 6
df_output.loc[mask, "inventory2_drift"] = (
    df_output.loc[mask, "last_inventory2_year"] - expected_year_inv2[mask]
)

# Drift Inventory3: esperado en tree_age=9 (2026 - PY + 9)
mask = df_output["last_inventory3_year"].notna()
expected_year_inv3 = df_output["planting_year"] + 9
df_output.loc[mask, "inventory3_drift"] = (
    df_output.loc[mask, "last_inventory3_year"] - expected_year_inv3[mask]
)

# ============================================================================
# 8. ORDENAR COLUMNAS FINALES
# ============================================================================
df_output = df_output[[
    "contract_code",
    "farmer_number",
    "contract_name",
    "planting_year",
    "status",
    "tree_age_2026",
    "age_classification_2026",
    "last_inventory1_year",
    "inventory1_drift",
    "last_inventory2_year",
    "inventory2_drift",
    "last_inventory3_year",
    "inventory3_drift",
    "inventory_2026",
    "expected_inventory_type_2026",
    "rule_applied"
]]

# ============================================================================
# 9. MOSTRAR RESULTADOS
# ============================================================================
print("=" * 80)
print("RESUMEN DEL PLAN DE INVENTARIOS 2026")
print("=" * 80)
print(f"Total contratos activos: {len(df_output)}")
print(f"Contratos que necesitan inventario en 2026: {df_output['inventory_2026'].sum()}")
print("\nDistribución por tipo de inventario esperado:")
print(df_output[df_output["inventory_2026"]]["expected_inventory_type_2026"].value_counts())

print("\n" + "=" * 80)
print("ANÁLISIS DE DRIFT (Inventarios realizados fuera del año esperado)")
print("=" * 80)
drift_cases = df_output[df_output["rule_applied"].str.contains("DRIFT", na=False)]
print(f"Contratos con drift detectado: {len(drift_cases)}")
if len(drift_cases) > 0:
    print("\nEjemplos de drift:")
    print(drift_cases[["contract_code", "planting_year", "last_inventory1_year",
                       "inventory1_drift", "rule_applied"]].head())

print("\n" + "=" * 80)
print("Contratos que SÍ necesitan inventario en 2026:")
print(df_output[df_output["inventory_2026"]].head(10))

💻 Conectado a la base de datos helloworldtree


C:\Users\HeyCe\AppData\Local\Temp\ipykernel_3588\3025748768.py:52: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["contract_code", "inventory_milestone"])["inventory_year"]


RESUMEN DEL PLAN DE INVENTARIOS 2026
Total contratos activos: 265
Contratos que necesitan inventario en 2026: 156

Distribución por tipo de inventario esperado:
expected_inventory_type_2026
Survival1     63
Inventory2    56
Inventory1    31
Survival0      6
Name: count, dtype: int64

ANÁLISIS DE DRIFT (Inventarios realizados fuera del año esperado)
Contratos con drift detectado: 0

Contratos que SÍ necesitan inventario en 2026:
   contract_code farmer_number                        contract_name  \
0         US0170         10132               Shady Branch Farms LLC   
3         US0166         10130                          Ken English   
4         US0167         10067         Mike Porter and Teri Galardi   
5         US0168         10133  Justin Montgomery and Rusty Elliott   
6         CR0112         20103           Manfred Goyenaga Rodriguez   
8         US0169         10134        Corley Walton Family Farm LLC   
9         MX0071         40066                   Emma Barrios Goméz   


In [1]:
from core.libs import pd, np
from core.db import get_engine

# ============================================================================
# 1. CARGAR DATOS BASE - COMBINAR AMBAS FUENTES
# ============================================================================
engine = get_engine()

print("=" * 80)
print("🔄 CARGANDO DATOS DE AMBAS FUENTES")
print("=" * 80)

# ============================================================================
# FUENTE 1: AÑOS 2021-2022 desde fact_contract_inventory_year
# ============================================================================
print("\n📥 Cargando 2021-2022 desde fact_contract_inventory_year...")
query_fact = """
SELECT
    contract_code,
    inventory_year,
    expected_inventory_type as inventory_type
FROM helloworldtree.inventory.fact_contract_inventory_year
WHERE observed_inventory = TRUE
  AND inventory_year IN (2021, 2022)
"""
df_fact_2021_2022 = pd.read_sql(query_fact, engine)
print(f"✅ Cargados {len(df_fact_2021_2022)} eventos de 2021-2022")
print(f"   Contratos únicos: {df_fact_2021_2022['contract_code'].nunique()}")

# ============================================================================
# FUENTE 2: AÑOS 2023-2025 desde Excel procesado
# ============================================================================
print("\n📥 Cargando 2023-2025 desde Excel...")
df_excel_2023_2025 = pd.read_excel(
    r"C:\Users\HeyCe\World Tree Technologies Inc\Operations - Documentos\WorldTreeSystem\DatabaseExports\inventories_tree_age.xlsx"
)
print(f"✅ Cargados {len(df_excel_2023_2025)} eventos de 2023-2025")
print(f"   Contratos únicos: {df_excel_2023_2025['contract_code'].nunique()}")

# ============================================================================
# COMBINAR AMBAS FUENTES
# ============================================================================
print("\n🔗 Combinando ambas fuentes...")

# Asegurar que ambas tienen las columnas necesarias
required_cols = ["contract_code", "inventory_year", "inventory_type"]

# df_fact ya tiene las columnas correctas
df_fact_clean = df_fact_2021_2022[required_cols].copy()

# df_excel necesita mantener sus columnas adicionales pero asegurar las básicas
df_excel_clean = df_excel_2023_2025.copy()

# Combinar
df_events = pd.concat([
    df_fact_clean,
    df_excel_clean
], ignore_index=True)

print(f"\n✅ DATOS COMBINADOS:")
print(f"   Total eventos: {len(df_events)}")
print(f"   Contratos únicos: {df_events['contract_code'].nunique()}")
print(f"   Años disponibles: {sorted(df_events['inventory_year'].dropna().unique())}")

# Distribución por año
print("\n📊 Eventos por año:")
year_dist = df_events.groupby("inventory_year").agg({
    "contract_code": ["count", "nunique"]
})
year_dist.columns = ["Total Eventos", "Contratos Únicos"]
print(year_dist)

# ============================================================================
# 2. CARGAR INFORMACIÓN DE CONTRATOS
# ============================================================================
query = """
SELECT
    cti.contract_code,
    cti.farmer_number,
    fpi.contract_name,
    cti.status
FROM masterdatabase.contract_tree_information cti
LEFT JOIN masterdatabase.farmer_personal_information fpi
    ON cti.farmer_number = fpi.farmer_number
"""
df_cti = pd.read_sql(query, engine)

# ============================================================================
# 3. CREAR TABLA PIVOT: CONTRATOS × AÑOS
# ============================================================================
print("\n" + "=" * 80)
print("📊 CREANDO TABLA PIVOT")
print("=" * 80)

# Filtrar solo eventos de tipo "Inventory"
inv_events = df_events[df_events["inventory_type"] == "Inventory"].copy()
print(f"\nEventos de tipo 'Inventory': {len(inv_events)}")

# Agrupar por contrato y año, marcando con "Yes" si hubo inventario
inventory_pivot = (
    inv_events
    .groupby(["contract_code", "inventory_year"])
    .size()
    .reset_index(name="count")
)

# Crear pivot table
inventory_pivot = inventory_pivot.pivot(
    index="contract_code",
    columns="inventory_year",
    values="count"
).notna()

# Reemplazar True/False con "Yes"/vacío
inventory_pivot = inventory_pivot.replace({True: "Yes", False: ""})

# ============================================================================
# 4. AGREGAR INFORMACIÓN DEL CONTRATO
# ============================================================================
# Merge con información básica del contrato
df_output = df_cti[["contract_code", "contract_name", "farmer_number", "status"]].merge(
    inventory_pivot.reset_index(),
    on="contract_code",
    how="left"
)

# Rellenar NaN con strings vacíos en las columnas de años
year_columns = [col for col in df_output.columns if isinstance(col, (int, float))]
for col in year_columns:
    df_output[col] = df_output[col].fillna("")

# Ordenar columnas: info del contrato + años en orden
info_cols = ["contract_code", "contract_name", "farmer_number", "status"]
year_cols = sorted([col for col in df_output.columns if col not in info_cols])
df_output = df_output[info_cols + year_cols]

# Ordenar por contract_code
df_output = df_output.sort_values("contract_code").reset_index(drop=True)

# ============================================================================
# 5. EXPORTAR A EXCEL
# ============================================================================
output_path = r"C:\Users\HeyCe\World Tree Technologies Inc\Operations - Documentos\WorldTreeSystem\DatabaseExports\inventory_history_pivot.xlsx"

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    df_output.to_excel(writer, sheet_name="Inventory History", index=False)

    # Formatear el sheet
    worksheet = writer.sheets["Inventory History"]

    # Ajustar ancho de columnas
    worksheet.column_dimensions['A'].width = 15  # contract_code
    worksheet.column_dimensions['B'].width = 35  # contract_name
    worksheet.column_dimensions['C'].width = 15  # farmer_number
    worksheet.column_dimensions['D'].width = 12  # status

    # Años: ancho fijo
    for col in range(5, len(df_output.columns) + 1):
        worksheet.column_dimensions[worksheet.cell(1, col).column_letter].width = 8

print("\n" + "=" * 80)
print("TABLA PIVOT DE INVENTARIOS POR AÑO")
print("=" * 80)
print(f"Total contratos: {len(df_output)}")
print(f"Rango de años: {min(year_cols)} - {max(year_cols)}")
print(f"\nArchivo exportado a:\n{output_path}")

# ============================================================================
# 6. ANÁLISIS DE CONTRATOS SIN INVENTARIOS
# ============================================================================
print("\n" + "=" * 80)
print("ANÁLISIS DE CONTRATOS SIN INVENTARIOS")
print("=" * 80)

# Identificar contratos sin ningún inventario registrado
contracts_with_inventory = df_output[year_cols].apply(lambda row: (row == "Yes").any(), axis=1)
df_output["has_any_inventory"] = contracts_with_inventory

df_no_inventory = df_output[~df_output["has_any_inventory"]].copy()

print(f"\n⚠️  Total contratos SIN inventarios: {len(df_no_inventory)}")
print(f"✅ Total contratos CON inventarios: {contracts_with_inventory.sum()}")

# Analizar por status
print("\n" + "-" * 80)
print("DISTRIBUCIÓN POR STATUS:")
print(df_no_inventory["status"].value_counts())

# Analizar contratos ACTIVOS sin inventario
df_active_no_inv = df_no_inventory[df_no_inventory["status"] == "Active"].copy()
print(f"\n🔴 Contratos ACTIVOS sin inventario: {len(df_active_no_inv)}")

if len(df_active_no_inv) > 0:
    print("\n⚠️  Primeros 20 contratos activos sin inventario:")
    print(df_active_no_inv[["contract_code", "contract_name", "status"]].head(20).to_string(index=False))

# ============================================================================
# 7. ESTADÍSTICAS DE INVENTARIOS POR AÑO
# ============================================================================
print("\n" + "=" * 80)
print("ESTADÍSTICAS DE INVENTARIOS POR AÑO")
print("=" * 80)

stats = []
for year in year_cols:
    count = (df_output[year] == "Yes").sum()
    stats.append({"Año": year, "Contratos con Inventario": count})

df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))

# ============================================================================
# 8. VERIFICACIÓN DE COBERTURA TEMPORAL
# ============================================================================
print("\n" + "=" * 80)
print("✅ VERIFICACIÓN FINAL")
print("=" * 80)
print(f"✅ Años 2021-2022: {(2021 in year_cols) and (2022 in year_cols)}")
print(f"✅ Años 2023-2025: {(2023 in year_cols) and (2024 in year_cols) and (2025 in year_cols)}")
print(f"✅ Cobertura completa: 2021-2025")

print("\n" + "=" * 80)
print("Primeras 10 filas:")
print(df_output.head(10).to_string())
print("=" * 80)

💻 Conectado a la base de datos helloworldtree
🔄 CARGANDO DATOS DE AMBAS FUENTES

📥 Cargando 2021-2022 desde fact_contract_inventory_year...
✅ Cargados 175 eventos de 2021-2022
   Contratos únicos: 143

📥 Cargando 2023-2025 desde Excel...
✅ Cargados 382 eventos de 2023-2025
   Contratos únicos: 240

🔗 Combinando ambas fuentes...

✅ DATOS COMBINADOS:
   Total eventos: 557
   Contratos únicos: 240
   Años disponibles: [np.float64(2021.0), np.float64(2022.0), np.float64(2023.0), np.float64(2024.0), np.float64(2025.0)]

📊 Eventos por año:
                Total Eventos  Contratos Únicos
inventory_year                                 
2021.0                    222               111
2022.0                    128                64
2023.0                      1                 1
2024.0                     95                93
2025.0                     88                87

📊 CREANDO TABLA PIVOT

Eventos de tipo 'Inventory': 161

TABLA PIVOT DE INVENTARIOS POR AÑO
Total contratos: 453
Rango de a

In [2]:
from core.libs import pd, np
from core.db import get_engine
from pathlib import Path
import re

# ============================================================================
# 1. CARGAR LOS 169 CONTRATOS ACTIVOS SIN INVENTARIO
# ============================================================================
engine = get_engine()

print("=" * 80)
print("🔍 DIAGNÓSTICO INTELIGENTE: 169 CONTRATOS SIN INVENTARIOS")
print("=" * 80)

# Cargar tabla pivot
df_pivot = pd.read_excel(
    r"C:\Users\HeyCe\World Tree Technologies Inc\Operations - Documentos\WorldTreeSystem\DatabaseExports\inventory_history_pivot.xlsx"
)

year_cols = [col for col in df_pivot.columns if isinstance(col, (int, float))]
contracts_with_inventory = df_pivot[year_cols].apply(lambda row: (row == "Yes").any(), axis=1)

df_missing = df_pivot[
    (~contracts_with_inventory) &
    (df_pivot["status"] == "Active")
].copy()

print(f"\n📊 Total contratos activos sin inventario: {len(df_missing)}")

# Obtener planting_year para análisis
query_planting = """
SELECT contract_code, planting_year
FROM masterdatabase.contract_tree_information
"""
df_planting = pd.read_sql(query_planting, engine)
df_missing = df_missing.merge(df_planting, on="contract_code", how="left")

# ============================================================================
# 2. DEFINIR RUTAS DE ARCHIVOS CONSOLIDADOS Y CARPETAS
# ============================================================================
base_path = Path(r"C:\Users\HeyCe\World Tree Technologies Inc\Forest Inventory - Documentos")

# Archivos consolidados 2021 (por país)
consolidated_2021 = {
    "USA": base_path / "USA" / "2021_ForestInventory" / "9-Database" / "2021_US_Database v1.1.xlsx",
    "Costa Rica": base_path / "Costa Rica" / "2021_ForestInventory" / "9-Database" / "2021_CR_Database v1.1.xlsx",
    "Guatemala": base_path / "Guatemala" / "2021_ForestInventory" / "9-Database" / "2021_GT_Database v1.1.xlsx",
    "Mexico": base_path / "Mexico" / "2021_ForestInventory" / "9-Database" / "2021_MX_Database v1.1.xlsx",
}

# Carpetas con individuales 2022
folders_2022 = {
    "USA": base_path / "USA" / "2022_ForestInventory" / "8-ForestMetrix_Projects",
    "Costa Rica": base_path / "Costa Rica" / "2022_ForestInventory" / "8-ForestMetrix_Projects",
    "Guatemala": base_path / "Guatemala" / "2022_ForestInventory" / "8-ForestMetrix_Projects",
    "Mexico": base_path / "Mexico" / "2022_ForestInventory" / "8-ForestMetrix_Projects",
}

# ============================================================================
# 3. FUNCIÓN PARA EXTRAER CONTRACT_CODE DE NOMBRE DE ARCHIVO
# ============================================================================
def extract_contract_code(filename):
    """Extrae US0001, CR0123, GT0045, MX0012 del nombre de archivo"""
    pattern = re.compile(r'(US|CR|GT|MX)\d{4}', re.IGNORECASE)
    matches = pattern.findall(filename.upper())
    return matches[0] if matches else None

# ============================================================================
# 4. VERIFICAR ARCHIVOS CONSOLIDADOS 2021
# ============================================================================
print("\n" + "=" * 80)
print("📁 VERIFICANDO ARCHIVOS CONSOLIDADOS 2021")
print("=" * 80)

contracts_in_2021 = {}
for country, file_path in consolidated_2021.items():
    if file_path.exists():
        print(f"\n✅ {country}: {file_path.name}")
        try:
            # Leer archivo consolidado
            df_2021 = pd.read_excel(file_path)
            print(f"   Columnas: {list(df_2021.columns)[:5]}...")

            # Buscar columna con contract_code
            contract_col = None
            for col in df_2021.columns:
                if 'contract' in str(col).lower() and 'code' in str(col).lower():
                    contract_col = col
                    break

            if contract_col:
                # Extraer contract codes
                codes = df_2021[contract_col].dropna().unique()
                contracts_in_2021[country] = [c[:6] for c in codes if isinstance(c, str)]
                print(f"   Contract codes encontrados: {len(contracts_in_2021[country])}")
            else:
                print(f"   ⚠️  No se encontró columna de contract_code")
        except Exception as e:
            print(f"   ❌ Error leyendo archivo: {e}")
    else:
        print(f"\n❌ {country}: Archivo no existe")

# ============================================================================
# 5. ESCANEAR CARPETAS 2022 (INDIVIDUALES)
# ============================================================================
print("\n" + "=" * 80)
print("📁 ESCANEANDO CARPETAS 2022 (ARCHIVOS INDIVIDUALES)")
print("=" * 80)

contracts_in_2022 = {}
for country, folder_path in folders_2022.items():
    if folder_path.exists():
        print(f"\n✅ {country}: {folder_path}")

        # Buscar todos los Excel en subcarpetas
        excel_files = list(folder_path.glob("**/*.xlsx")) + list(folder_path.glob("**/*.xls"))
        excel_files = [f for f in excel_files if not f.name.startswith("~$")]

        # Extraer contract codes
        codes = []
        for file in excel_files:
            code = extract_contract_code(file.name)
            if code:
                codes.append(code)

        contracts_in_2022[country] = list(set(codes))
        print(f"   Archivos Excel: {len(excel_files)}")
        print(f"   Contract codes únicos: {len(contracts_in_2022[country])}")
    else:
        print(f"\n❌ {country}: Carpeta no existe")

# ============================================================================
# 6. VERIFICAR QUÉ YA SE PROCESÓ (2023-2025)
# ============================================================================
print("\n" + "=" * 80)
print("📊 VERIFICANDO DATOS YA PROCESADOS (2023-2025)")
print("=" * 80)

# Cargar metadata procesado
df_metadata = pd.read_excel("summary_metadata_ALL_BATCHES.xlsx")
contracts_processed_2023_2025 = df_metadata["contract_code"].unique()
print(f"✅ Contract codes en metadata 2023-2025: {len(contracts_processed_2023_2025)}")

# ============================================================================
# 7. ANÁLISIS CRUZADO POR AÑO
# ============================================================================
print("\n" + "=" * 80)
print("🔍 ANÁLISIS CRUZADO: 169 CONTRATOS VS ARCHIVOS DISPONIBLES")
print("=" * 80)

df_missing["contract_short"] = df_missing["contract_code"].str[:6]

# Combinar todos los contracts encontrados
all_contracts_2021 = []
for codes in contracts_in_2021.values():
    all_contracts_2021.extend(codes)

all_contracts_2022 = []
for codes in contracts_in_2022.values():
    all_contracts_2022.extend(codes)

results = []
for _, row in df_missing.iterrows():
    contract = row["contract_short"]
    planting_year = row["planting_year"]

    # Determinar qué archivos deberían existir según planting_year
    expected_years = []
    if planting_year <= 2020:
        expected_years = [2021, 2022, 2023, 2024, 2025]
    elif planting_year == 2021:
        expected_years = [2022, 2023, 2024, 2025]
    elif planting_year == 2022:
        expected_years = [2023, 2024, 2025]
    elif planting_year == 2023:
        expected_years = [2024, 2025]
    elif planting_year == 2024:
        expected_years = [2025]
    elif planting_year == 2025:
        expected_years = []

    # Verificar en qué años existe
    found_in_2021 = contract in all_contracts_2021
    found_in_2022 = contract in all_contracts_2022
    found_in_2023_2025 = contract in contracts_processed_2023_2025

    # Diagnóstico
    if len(expected_years) == 0:
        status = "RECIÉN_PLANTADO_2025"
        detail = "No esperado en ningún año todavía"
    elif found_in_2021 or found_in_2022 or found_in_2023_2025:
        status = "ARCHIVO_EXISTE_NO_PROCESADO"
        locations = []
        if found_in_2021:
            locations.append("2021_consolidado")
        if found_in_2022:
            locations.append("2022_individual")
        if found_in_2023_2025:
            locations.append("2023-2025_metadata")
        detail = f"Encontrado en: {', '.join(locations)}"
    else:
        status = "NO_EXISTE_ARCHIVO"
        detail = f"Esperado en: {expected_years}"

    results.append({
        "contract_code": contract,
        "contract_name": row["contract_name"],
        "planting_year": planting_year,
        "expected_years": str(expected_years),
        "status": status,
        "found_2021": found_in_2021,
        "found_2022": found_in_2022,
        "found_2023_2025": found_in_2023_2025,
        "detail": detail
    })

df_diagnostic = pd.DataFrame(results)

# ============================================================================
# 8. RESUMEN POR CATEGORÍA
# ============================================================================
print("\n📊 RESUMEN POR CATEGORÍA:")
print("=" * 80)
print(df_diagnostic["status"].value_counts())

# Casos críticos: archivos existen pero no se procesaron
df_critical = df_diagnostic[df_diagnostic["status"] == "ARCHIVO_EXISTE_NO_PROCESADO"]
if len(df_critical) > 0:
    print(f"\n🚨 {len(df_critical)} CASOS CRÍTICOS: Archivos existen pero NO se procesaron")
    print("\nPrimeros 20 casos:")
    print(df_critical[["contract_code", "contract_name", "planting_year", "detail"]].head(20).to_string(index=False))

# Recién plantados 2025
df_recent = df_diagnostic[df_diagnostic["status"] == "RECIÉN_PLANTADO_2025"]
if len(df_recent) > 0:
    print(f"\n✅ {len(df_recent)} contratos recién plantados en 2025 (normal que no tengan inventario)")

# Sin archivos
df_no_files = df_diagnostic[df_diagnostic["status"] == "NO_EXISTE_ARCHIVO"]
if len(df_no_files) > 0:
    print(f"\n❌ {len(df_no_files)} contratos SIN archivos (nunca se hizo cruise)")
    print("\nPrimeros 10:")
    print(df_no_files[["contract_code", "contract_name", "planting_year", "expected_years"]].head(10).to_string(index=False))

# ============================================================================
# 9. EXPORTAR DIAGNÓSTICO
# ============================================================================
output_path = r"C:\Users\HeyCe\World Tree Technologies Inc\Operations - Documentos\WorldTreeSystem\DatabaseExports\diagnostic_missing_inventories_smart.xlsx"

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Resumen
    summary = pd.DataFrame([
        {"Categoría": "Total sin inventario", "Cantidad": len(df_diagnostic)},
        {"Categoría": "Archivos existen (NO procesados)", "Cantidad": len(df_critical)},
        {"Categoría": "Recién plantados 2025", "Cantidad": len(df_recent)},
        {"Categoría": "Sin archivos (no hay cruise)", "Cantidad": len(df_no_files)},
        {"Categoría": "Encontrados en 2021", "Cantidad": df_diagnostic["found_2021"].sum()},
        {"Categoría": "Encontrados en 2022", "Cantidad": df_diagnostic["found_2022"].sum()},
        {"Categoría": "Encontrados en 2023-2025", "Cantidad": df_diagnostic["found_2023_2025"].sum()},
    ])
    summary.to_excel(writer, sheet_name="Resumen", index=False)

    # Diagnóstico completo
    df_diagnostic.to_excel(writer, sheet_name="Diagnostic", index=False)

    # Casos críticos
    if len(df_critical) > 0:
        df_critical.to_excel(writer, sheet_name="Critical_Cases", index=False)

    # Sin archivos
    if len(df_no_files) > 0:
        df_no_files.to_excel(writer, sheet_name="No_Files", index=False)

print(f"\n💾 Diagnóstico exportado a:")
print(f"   {output_path}")
print("\n" + "=" * 80)
print("✅ DIAGNÓSTICO COMPLETO")
print("=" * 80)

💻 Conectado a la base de datos helloworldtree
🔍 DIAGNÓSTICO INTELIGENTE: 169 CONTRATOS SIN INVENTARIOS

📊 Total contratos activos sin inventario: 169

📁 VERIFICANDO ARCHIVOS CONSOLIDADOS 2021

✅ USA: 2021_US_Database v1.1.xlsx
   Columnas: ['Property', 'PlantingYear', 'GradingsBA', 'GradingsBAc', 'GradingsDBH']...
   ⚠️  No se encontró columna de contract_code

❌ Costa Rica: Archivo no existe

❌ Guatemala: Archivo no existe

❌ Mexico: Archivo no existe

📁 ESCANEANDO CARPETAS 2022 (ARCHIVOS INDIVIDUALES)

✅ USA: C:\Users\HeyCe\World Tree Technologies Inc\Forest Inventory - Documentos\USA\2022_ForestInventory\8-ForestMetrix_Projects
   Archivos Excel: 68
   Contract codes únicos: 1

❌ Costa Rica: Carpeta no existe

❌ Guatemala: Carpeta no existe

❌ Mexico: Carpeta no existe

📊 VERIFICANDO DATOS YA PROCESADOS (2023-2025)
✅ Contract codes en metadata 2023-2025: 175

🔍 ANÁLISIS CRUZADO: 169 CONTRATOS VS ARCHIVOS DISPONIBLES

📊 RESUMEN POR CATEGORÍA:
status
NO_EXISTE_ARCHIVO              84


In [3]:
from core.libs import pd, np
from core.db import get_engine
import re

print("=" * 80)
print("🚀 PIPELINE COMPLETO: CORREGIR Y REGENERAR")
print("=" * 80)

# ============================================================================
# PASO 1: CARGAR Y REPARAR METADATA
# ============================================================================
print("\n📥 PASO 1: Cargando metadata...")

df_metadata = pd.read_excel("summary_metadata_ALL_BATCHES.xlsx")
print(f"✅ Cargados {len(df_metadata)} registros")
print(f"⚠️  Registros sin inventory_year: {df_metadata['inventory_year'].isna().sum()}")

# ============================================================================
# FUNCIÓN: EXTRAER AÑO DEL source_file
# ============================================================================
def extract_year_from_source(source_file):
    """Extrae año del path: 'USA/2023_ForestInventory' → 2023"""
    if pd.isna(source_file):
        return None

    pattern = r'(20\d{2})'
    matches = re.findall(pattern, str(source_file))
    return int(matches[0]) if matches else None

# ============================================================================
# REPARAR inventory_year USANDO source_file
# ============================================================================
print("\n🔧 PASO 2: Reparando inventory_year faltantes...")

df_fixed = df_metadata.copy()
mask_missing = df_fixed["inventory_year"].isna()

# Extraer año del source_file
df_fixed.loc[mask_missing, "year_from_source"] = df_fixed.loc[mask_missing, "source_file"].apply(
    extract_year_from_source
)

# Rellenar con año extraído
before = mask_missing.sum()
df_fixed.loc[mask_missing, "inventory_year"] = df_fixed.loc[mask_missing, "year_from_source"]
after = df_fixed["inventory_year"].isna().sum()

print(f"✅ Reparados: {before - after} registros usando source_file")
print(f"⚠️  Todavía faltantes: {after} registros")

# ============================================================================
# RECALCULAR tree_age
# ============================================================================
df_fixed["tree_age"] = df_fixed["inventory_year"] - df_fixed["planting_year"]

# ============================================================================
# GUARDAR METADATA CORREGIDO
# ============================================================================
output_metadata = "summary_metadata_ALL_BATCHES_FIXED.xlsx"
df_fixed.to_excel(output_metadata, index=False)
print(f"\n💾 Metadata corregido guardado: {output_metadata}")

# ============================================================================
# PASO 3: CARGAR DATOS DE 2021-2022 DESDE FACT TABLE
# ============================================================================
print("\n📥 PASO 3: Cargando 2021-2022 desde fact_table...")

engine = get_engine()

query_fact = """
SELECT
    contract_code,
    inventory_year,
    expected_inventory_type as inventory_type
FROM helloworldtree.inventory.fact_contract_inventory_year
WHERE observed_inventory = TRUE
  AND inventory_year IN (2021, 2022)
"""
df_fact_2021_2022 = pd.read_sql(query_fact, engine)

print(f"✅ {len(df_fact_2021_2022)} eventos de 2021-2022")
print(f"   Contratos únicos: {df_fact_2021_2022['contract_code'].nunique()}")

# ============================================================================
# PASO 4: COMBINAR 2021-2022 + 2023-2025 CORREGIDO
# ============================================================================
print("\n🔗 PASO 4: Combinando todas las fuentes...")

# Preparar datos de fact (2021-2022)
df_fact_clean = df_fact_2021_2022[["contract_code", "inventory_year", "inventory_type"]].copy()

# Preparar datos de metadata corregido (2023-2025)
df_meta_clean = df_fixed[["contract_code", "inventory_year", "inventory_type"]].copy()

# Combinar
df_events_combined = pd.concat([
    df_fact_clean,
    df_meta_clean
], ignore_index=True)

print(f"✅ Total eventos combinados: {len(df_events_combined)}")
print(f"   Contratos únicos: {df_events_combined['contract_code'].nunique()}")
print(f"   Años: {sorted(df_events_combined['inventory_year'].dropna().unique())}")

# Distribución por año
print("\n📊 Eventos por año:")
year_dist = df_events_combined.groupby("inventory_year").agg({
    "contract_code": ["count", "nunique"]
})
year_dist.columns = ["Total Eventos", "Contratos Únicos"]
print(year_dist)

# ============================================================================
# PASO 5: FILTRAR SOLO INVENTORY TYPE
# ============================================================================
print("\n🔍 PASO 5: Filtrando tipo 'Inventory'...")

# Normalizar inventory_type (algunos tienen espacios o mayúsculas)
df_events_combined["inventory_type_clean"] = (
    df_events_combined["inventory_type"]
    .str.strip()
    .str.lower()
)

# Filtrar solo "inventory" (no Survival0, Survival1, etc.)
mask_inventory = df_events_combined["inventory_type_clean"] == "inventory"
df_inventory_only = df_events_combined[mask_inventory].copy()

print(f"✅ Eventos tipo 'Inventory': {len(df_inventory_only)}")
print(f"   Contratos únicos: {df_inventory_only['contract_code'].nunique()}")

# ============================================================================
# PASO 6: CREAR TABLA PIVOT
# ============================================================================
print("\n📊 PASO 6: Creando tabla pivot...")

# Agrupar por contrato y año
inventory_pivot = (
    df_inventory_only
    .groupby(["contract_code", "inventory_year"])
    .size()
    .reset_index(name="count")
)

# Crear pivot
inventory_pivot = inventory_pivot.pivot(
    index="contract_code",
    columns="inventory_year",
    values="count"
).notna()

# Reemplazar True/False con "Yes"/vacío
inventory_pivot = inventory_pivot.replace({True: "Yes", False: ""})

# ============================================================================
# PASO 7: AGREGAR INFO DE CONTRATOS
# ============================================================================
print("\n📋 PASO 7: Agregando información de contratos...")

query_contracts = """
SELECT
    cti.contract_code,
    cti.farmer_number,
    fpi.contract_name,
    cti.status,
    cti.planting_year
FROM masterdatabase.contract_tree_information cti
LEFT JOIN masterdatabase.farmer_personal_information fpi
    ON cti.farmer_number = fpi.farmer_number
"""
df_contracts = pd.read_sql(query_contracts, engine)

# Merge
df_output = df_contracts[["contract_code", "contract_name", "farmer_number", "status", "planting_year"]].merge(
    inventory_pivot.reset_index(),
    on="contract_code",
    how="left"
)

# Rellenar NaN con vacío
year_columns = [col for col in df_output.columns if isinstance(col, (int, float))]
for col in year_columns:
    df_output[col] = df_output[col].fillna("")

# Ordenar columnas
info_cols = ["contract_code", "contract_name", "farmer_number", "status", "planting_year"]
year_cols = sorted([col for col in df_output.columns if col not in info_cols])
df_output = df_output[info_cols + year_cols]

# Ordenar por contract_code
df_output = df_output.sort_values("contract_code").reset_index(drop=True)

# ============================================================================
# PASO 8: ANÁLISIS DE CONTRATOS SIN INVENTARIOS
# ============================================================================
print("\n📊 PASO 8: Analizando contratos sin inventarios...")

contracts_with_inventory = df_output[year_cols].apply(lambda row: (row == "Yes").any(), axis=1)
df_output["has_any_inventory"] = contracts_with_inventory

df_no_inventory = df_output[~df_output["has_any_inventory"]].copy()
df_active_no_inv = df_no_inventory[df_no_inventory["status"] == "Active"]

print(f"\n⚠️  Total contratos SIN inventarios: {len(df_no_inventory)}")
print(f"✅ Total contratos CON inventarios: {contracts_with_inventory.sum()}")
print(f"🔴 Contratos ACTIVOS sin inventario: {len(df_active_no_inv)}")

# Comparación con antes
print(f"\n📉 MEJORA:")
print(f"   Antes: 169 contratos activos sin inventario")
print(f"   Ahora: {len(df_active_no_inv)} contratos activos sin inventario")
print(f"   🎉 Recuperados: {169 - len(df_active_no_inv)} contratos!")

# ============================================================================
# PASO 9: EXPORTAR TABLA PIVOT FINAL
# ============================================================================
output_pivot = r"C:\Users\HeyCe\World Tree Technologies Inc\Operations - Documentos\WorldTreeSystem\DatabaseExports\inventory_history_pivot_FIXED.xlsx"

with pd.ExcelWriter(output_pivot, engine='openpyxl') as writer:
    df_output.to_excel(writer, sheet_name="Inventory History", index=False)

    # Formatear
    worksheet = writer.sheets["Inventory History"]
    worksheet.column_dimensions['A'].width = 15
    worksheet.column_dimensions['B'].width = 35
    worksheet.column_dimensions['C'].width = 15
    worksheet.column_dimensions['D'].width = 12
    worksheet.column_dimensions['E'].width = 12

    for col in range(6, len(df_output.columns) + 1):
        worksheet.column_dimensions[worksheet.cell(1, col).column_letter].width = 8

print(f"\n💾 Tabla pivot CORREGIDA exportada a:")
print(f"   {output_pivot}")

# ============================================================================
# PASO 10: ESTADÍSTICAS FINALES
# ============================================================================
print("\n" + "=" * 80)
print("📊 ESTADÍSTICAS FINALES")
print("=" * 80)
print(f"Total contratos: {len(df_output)}")
print(f"Contratos con inventarios: {contracts_with_inventory.sum()}")
print(f"Rango de años: {min(year_cols)} - {max(year_cols)}")

print("\n📊 Inventarios por año:")
stats = []
for year in year_cols:
    count = (df_output[year] == "Yes").sum()
    stats.append({"Año": year, "Contratos": count})
print(pd.DataFrame(stats).to_string(index=False))

if len(df_active_no_inv) > 0:
    print(f"\n⚠️  {len(df_active_no_inv)} contratos activos todavía sin inventario:")
    print("\nAnálisis por planting_year:")
    print(df_active_no_inv.groupby("planting_year").size().sort_index())

print("\n" + "=" * 80)
print("✅ PIPELINE COMPLETO - TODO LISTO")
print("=" * 80)
print("\n📋 Archivos generados:")
print(f"   1. {output_metadata}")
print(f"   2. {output_pivot}")
print("\n🎯 Usa inventory_history_pivot_FIXED.xlsx para Drake")

🚀 PIPELINE COMPLETO: CORREGIR Y REGENERAR

📥 PASO 1: Cargando metadata...
✅ Cargados 207 registros
⚠️  Registros sin inventory_year: 23

🔧 PASO 2: Reparando inventory_year faltantes...
✅ Reparados: 23 registros usando source_file
⚠️  Todavía faltantes: 0 registros

💾 Metadata corregido guardado: summary_metadata_ALL_BATCHES_FIXED.xlsx

📥 PASO 3: Cargando 2021-2022 desde fact_table...
💻 Conectado a la base de datos helloworldtree
✅ 175 eventos de 2021-2022
   Contratos únicos: 143

🔗 PASO 4: Combinando todas las fuentes...
✅ Total eventos combinados: 382
   Contratos únicos: 240
   Años: [np.float64(2021.0), np.float64(2022.0), np.float64(2023.0), np.float64(2024.0), np.float64(2025.0)]

📊 Eventos por año:
                Total Eventos  Contratos Únicos
inventory_year                                 
2021.0                    111               111
2022.0                     64                64
2023.0                     14                13
2024.0                    105               1

In [4]:
from core.libs import pd, np
from core.db import get_engine

print("=" * 80)
print("🎯 CREANDO DOS TABLAS PIVOT")
print("=" * 80)
print("Tabla 1: General - ¿Se hizo algo? (Yes/No)")
print("Tabla 2: Detallada - ¿Qué tipo específico?")
print("=" * 80)

# ============================================================================
# 1. CARGAR TODOS LOS DATOS
# ============================================================================
engine = get_engine()

# 2021-2022 desde fact
query_fact = """
SELECT
    contract_code,
    inventory_year,
    expected_inventory_type as inventory_type
FROM helloworldtree.inventory.fact_contract_inventory_year
WHERE observed_inventory = TRUE
  AND inventory_year IN (2021, 2022)
"""
df_fact = pd.read_sql(query_fact, engine)

# 2023-2025 desde metadata corregido
df_metadata = pd.read_excel("summary_metadata_ALL_BATCHES_FIXED.xlsx")

# Combinar
df_all_events = pd.concat([
    df_fact[["contract_code", "inventory_year", "inventory_type"]],
    df_metadata[["contract_code", "inventory_year", "inventory_type"]]
], ignore_index=True)

print(f"\n✅ Total eventos: {len(df_all_events)}")
print(f"✅ Contratos únicos: {df_all_events['contract_code'].nunique()}")
print(f"✅ Años: {sorted(df_all_events['inventory_year'].dropna().unique())}")

# Distribución por tipo
print("\n📊 Distribución por inventory_type:")
print(df_all_events["inventory_type"].value_counts())

# ============================================================================
# 2. CARGAR INFO DE CONTRATOS
# ============================================================================
query_contracts = """
SELECT
    cti.contract_code,
    cti.farmer_number,
    fpi.contract_name,
    cti.status,
    cti.planting_year
FROM masterdatabase.contract_tree_information cti
LEFT JOIN masterdatabase.farmer_personal_information fpi
    ON cti.farmer_number = fpi.farmer_number
"""
df_contracts = pd.read_sql(query_contracts, engine)

# ============================================================================
# 3. TABLA 1: GENERAL - YES/NO (INCLUYE TODO)
# ============================================================================
print("\n" + "=" * 80)
print("📊 TABLA 1: GENERAL (Yes/No)")
print("=" * 80)

# Crear pivot: cualquier evento = "Yes"
pivot_general = (
    df_all_events
    .groupby(["contract_code", "inventory_year"])
    .size()
    .reset_index(name="count")
)

pivot_general = pivot_general.pivot(
    index="contract_code",
    columns="inventory_year",
    values="count"
).notna()

# Convertir a Yes/vacío
pivot_general = pivot_general.replace({True: "Yes", False: ""})

# Merge con info de contratos
df_table1 = df_contracts[["contract_code", "contract_name", "farmer_number", "status", "planting_year"]].merge(
    pivot_general.reset_index(),
    on="contract_code",
    how="left"
)

# Rellenar NaN con vacío
year_columns = [col for col in df_table1.columns if isinstance(col, (int, float))]
for col in year_columns:
    df_table1[col] = df_table1[col].fillna("")

# Ordenar columnas
info_cols = ["contract_code", "contract_name", "farmer_number", "status", "planting_year"]
year_cols = sorted([col for col in df_table1.columns if col not in info_cols])
df_table1 = df_table1[info_cols + year_cols]
df_table1 = df_table1.sort_values("contract_code").reset_index(drop=True)

# Estadísticas Tabla 1
contracts_with_any = df_table1[year_cols].apply(lambda row: (row == "Yes").any(), axis=1)
df_table1["has_any_event"] = contracts_with_any
active_without = df_table1[(~contracts_with_any) & (df_table1["status"] == "Active")]

print(f"✅ Contratos con algún evento: {contracts_with_any.sum()}")
print(f"⚠️  Contratos sin eventos: {(~contracts_with_any).sum()}")
print(f"🔴 Contratos ACTIVOS sin eventos: {len(active_without)}")

# ============================================================================
# 4. TABLA 2: DETALLADA - MUESTRA EL TIPO
# ============================================================================
print("\n" + "=" * 80)
print("📊 TABLA 2: DETALLADA (Muestra tipo específico)")
print("=" * 80)

# Para cada contrato-año, obtener el tipo (si hay múltiples, tomar el primero)
pivot_detailed = (
    df_all_events
    .sort_values("inventory_year")  # Ordenar por año
    .groupby(["contract_code", "inventory_year"])["inventory_type"]
    .first()  # Tomar el primer tipo si hay múltiples en el mismo año
    .reset_index()
)

pivot_detailed = pivot_detailed.pivot(
    index="contract_code",
    columns="inventory_year",
    values="inventory_type"
)

# Rellenar NaN con vacío
pivot_detailed = pivot_detailed.fillna("")

# Merge con info de contratos
df_table2 = df_contracts[["contract_code", "contract_name", "farmer_number", "status", "planting_year"]].merge(
    pivot_detailed.reset_index(),
    on="contract_code",
    how="left"
)

# Rellenar NaN con vacío
year_columns_2 = [col for col in df_table2.columns if isinstance(col, (int, float))]
for col in year_columns_2:
    df_table2[col] = df_table2[col].fillna("")

# Ordenar columnas
df_table2 = df_table2[info_cols + year_cols]
df_table2 = df_table2.sort_values("contract_code").reset_index(drop=True)

# Estadísticas Tabla 2
print(f"✅ Contratos en tabla detallada: {len(df_table2)}")

# ============================================================================
# 5. EXPORTAR AMBAS TABLAS A UN SOLO EXCEL
# ============================================================================
output_path = r"C:\Users\HeyCe\World Tree Technologies Inc\Operations - Documentos\WorldTreeSystem\DatabaseExports\inventory_history_BOTH_TABLES.xlsx"

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # TABLA 1: General (Yes/No)
    df_table1_export = df_table1.drop(columns=["has_any_event"])
    df_table1_export.to_excel(writer, sheet_name="1_General_YesNo", index=False)

    # TABLA 2: Detallada (Tipos)
    df_table2.to_excel(writer, sheet_name="2_Detailed_Types", index=False)

    # RESUMEN
    summary_data = {
        "Métrica": [
            "Total contratos en sistema",
            "Contratos con algún evento (Tabla 1)",
            "Contratos sin eventos (Tabla 1)",
            "Contratos ACTIVOS sin eventos",
            "Total eventos registrados",
            "Rango de años",
        ],
        "Valor": [
            len(df_table1),
            contracts_with_any.sum(),
            (~contracts_with_any).sum(),
            len(active_without),
            len(df_all_events),
            f"{min(year_cols)} - {max(year_cols)}"
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name="0_Resumen", index=False)

    # Formatear columnas en ambas sheets
    for sheet_name in ["1_General_YesNo", "2_Detailed_Types"]:
        worksheet = writer.sheets[sheet_name]
        worksheet.column_dimensions['A'].width = 15
        worksheet.column_dimensions['B'].width = 35
        worksheet.column_dimensions['C'].width = 15
        worksheet.column_dimensions['D'].width = 12
        worksheet.column_dimensions['E'].width = 12

        for col in range(6, len(info_cols) + len(year_cols) + 1):
            worksheet.column_dimensions[worksheet.cell(1, col).column_letter].width = 10

print(f"\n💾 Ambas tablas exportadas a:")
print(f"   {output_path}")

# ============================================================================
# 6. RESUMEN FINAL
# ============================================================================
print("\n" + "=" * 80)
print("📊 RESUMEN FINAL")
print("=" * 80)

print(f"\n📋 TABLA 1 - GENERAL (Yes/No):")
print(f"   • Incluye: TODO (Inventory, Survival, Permanent, etc.)")
print(f"   • Contratos con eventos: {contracts_with_any.sum()}")
print(f"   • Contratos sin eventos: {(~contracts_with_any).sum()}")
print(f"   • Activos sin eventos: {len(active_without)}")

print(f"\n📋 TABLA 2 - DETALLADA (Tipos):")
print(f"   • Muestra el inventory_type específico")
print(f"   • Permite ver Inventory vs Survival vs Permanent")
print(f"   • Mismo número de contratos: {len(df_table2)}")

print(f"\n📊 Eventos por año:")
for year in year_cols:
    count_table1 = (df_table1[year] == "Yes").sum()
    count_table2 = (df_table2[year] != "").sum()
    print(f"   {int(year)}: {count_table1} contratos con eventos")

if len(active_without) > 0:
    print(f"\n⚠️  {len(active_without)} contratos ACTIVOS sin ningún evento:")
    print("\n   Por planting_year:")
    print(active_without.groupby("planting_year").size().sort_index().to_string())

print("\n" + "=" * 80)
print("✅ ¡TU ALMA PUEDE DESCANSAR! 🕊️")
print("=" * 80)
print("\n🎯 Archivos listos para Drake:")
print("   Sheet 1: General (Yes/No) - Vista rápida")
print("   Sheet 2: Detallada (Tipos) - Vista completa")
print("   Sheet 3: Resumen - Estadísticas")

🎯 CREANDO DOS TABLAS PIVOT
Tabla 1: General - ¿Se hizo algo? (Yes/No)
Tabla 2: Detallada - ¿Qué tipo específico?
💻 Conectado a la base de datos helloworldtree

✅ Total eventos: 382
✅ Contratos únicos: 240
✅ Años: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

📊 Distribución por inventory_type:
inventory_type
Inventory               131
Survival1                83
Survival0                76
Permanent                47
Survival 1               20
Survival 0               11
0                        10
12                        1
Árboles Contratados:      1
RFT                       1
inventory                 1
Name: count, dtype: int64

📊 TABLA 1: GENERAL (Yes/No)
✅ Contratos con algún evento: 238
⚠️  Contratos sin eventos: 215
🔴 Contratos ACTIVOS sin eventos: 87

📊 TABLA 2: DETALLADA (Muestra tipo específico)
✅ Contratos en tabla detallada: 453

💾 Ambas tablas exportadas a:
   C:\Users\HeyCe\World Tree Technologies Inc\Operations - Documentos\WorldTr